In [1]:
import functools
import jax
from jax import lax
from jax import numpy as jnp
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu
import numpy as np

P = jax.sharding.PartitionSpec

num_devices = jax.local_device_count()
assert num_devices > 1, "Please run this notebook with more than one device."
assert "TPU" in jax.devices()[0].device_kind, "Please run this notebook with TPU devices."
print(f"Running with {num_devices} {jax.devices()[0].device_kind} devices.")

/home/lsiyuan_google_com/miniconda3/envs/torch312/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Running with 8 TPU v6 lite devices.


In [2]:
partition = P('x', None)
mesh = jax.make_mesh((num_devices,), ('x',))
sharding = jax.sharding.NamedSharding(mesh, partition)

n_local_tokens = 16
n_local_col = 128

x = jnp.arange(num_devices * n_local_tokens * n_local_col).reshape(num_devices * n_local_tokens, n_local_col)
x = jax.device_put(x, sharding)
print(x.addressable_shards)
# print(x.dtype)
# print(x.shape)
x_shape = x.shape

[Shard(device=TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), index=(slice(0, 16, None), slice(None, None, None)), replica_id=0, data=[[   0    1    2 ...  125  126  127]
 [ 128  129  130 ...  253  254  255]
 [ 256  257  258 ...  381  382  383]
 ...
 [1664 1665 1666 ... 1789 1790 1791]
 [1792 1793 1794 ... 1917 1918 1919]
 [1920 1921 1922 ... 2045 2046 2047]]), Shard(device=TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), index=(slice(16, 32, None), slice(None, None, None)), replica_id=0, data=[[2048 2049 2050 ... 2173 2174 2175]
 [2176 2177 2178 ... 2301 2302 2303]
 [2304 2305 2306 ... 2429 2430 2431]
 ...
 [3712 3713 3714 ... 3837 3838 3839]
 [3840 3841 3842 ... 3965 3966 3967]
 [3968 3969 3970 ... 4093 4094 4095]]), Shard(device=TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), index=(slice(32, 48, None), slice(None, None, None)), replica_id=0, data=[[4096 4097 4098 ... 4221 4222 4223]
 [4224 4225 4226 ... 4349 4350 4351]
 [4352 435

In [3]:
# Transfer target
targets = jnp.arange(num_devices).reshape(-1, 1)
print(targets)
targets = jnp.concatenate([targets[1:], targets[0:1]])
print(targets)
targets = jnp.repeat(targets, n_local_tokens)
print(targets)
targets = jax.device_put(targets, jax.sharding.NamedSharding(mesh, P('x')))

[[0]
 [1]
 [2]
 [3]
 [4]
 [5]
 [6]
 [7]]
[[1]
 [2]
 [3]
 [4]
 [5]
 [6]
 [7]
 [0]]
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 7 7 7 7 7 7 7 7 7 7 7 7 7 7 7
 7 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## Note

`token-expert_idx`: `[T*top_k]`

`x`: `[T*top_k, D]`

TODO

Use HBM as input

In [4]:
def send_recv_kernel(target_ref, input_ref, output_ref, send_sem, recv_sem):
    my_id = lax.axis_index('x')
    tok_idx = pl.program_id(0)
    blk_idx = pl.program_id(1)
    target = target_ref[blk_idx * 8 + tok_idx]
    # target = target_ref[tok_idx]
    # target = target_ref[blk_idx]
    remote_copy_op = pltpu.make_async_remote_copy(
        src_ref=input_ref.at[pl.ds(tok_idx, 1)],
        dst_ref=output_ref.at[pl.ds(tok_idx, 1)],
        send_sem=send_sem,
        recv_sem=recv_sem,
        device_id=(target,),
        device_id_type=pltpu.DeviceIdType.MESH,
    )
    remote_copy_op.start()
    remote_copy_op.wait()


out_shape = jax.ShapeDtypeStruct((n_local_tokens, n_local_col), jnp.int32)
token_block_size = n_local_tokens // 2
x_block_shape = (token_block_size, n_local_col)
grid_spec = pltpu.PrefetchScalarGridSpec(
    num_scalar_prefetch=1,
    in_specs=[
        pl.BlockSpec(x_block_shape, lambda i, j, k : (j, 0)),
    ],
    out_specs=pl.BlockSpec(x_block_shape, lambda i, j, k : (j, 0)),
    scratch_shapes=(
        [pltpu.SemaphoreType.DMA] * 2),
    grid=(token_block_size, n_local_tokens // x_block_shape[0]),
)
right_permute = pl.pallas_call(
    # functools.partial(send_recv_kernel, token_block_size=token_block_size),
    send_recv_kernel,
    out_shape=out_shape,
    grid_spec=grid_spec,
)
# Wrap the kernel within a shard_map to call.
pallas_result = jax.jit(
    jax.shard_map(
        right_permute,
        mesh=mesh,
        in_specs=(P('x'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False,
    )
)(targets, x)

# Compare Pallas result to XLA shard_map result.
perm = tuple((src, (src + 1) % num_devices) for src in range(num_devices))

xla_result = jax.jit(
    jax.shard_map(
        lambda x: lax.ppermute(x, 'x', perm),
        mesh=mesh, in_specs=P('x', None), out_specs=P('x', None))
)(x)

print(pallas_result)

np.testing.assert_array_equal(pallas_result, xla_result)

[[14336 14337 14338 ... 14461 14462 14463]
 [14464 14465 14466 ... 14589 14590 14591]
 [14592 14593 14594 ... 14717 14718 14719]
 ...
 [13952 13953 13954 ... 14077 14078 14079]
 [14080 14081 14082 ... 14205 14206 14207]
 [14208 14209 14210 ... 14333 14334 14335]]


In [ ]:
targets.dtype

In [5]:
targets.addressable_shards

[Shard(device=TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), index=(slice(0, 16, None),), replica_id=0, data=[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]),
 Shard(device=TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), index=(slice(16, 32, None),), replica_id=0, data=[2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]),
 Shard(device=TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), index=(slice(32, 48, None),), replica_id=0, data=[3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3]),
 Shard(device=TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), index=(slice(48, 64, None),), replica_id=0, data=[4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4]),
 Shard(device=TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0), index=(slice(64, 80, None),), replica_id=0, data=[5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5]),
 Shard(device=TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), index=(slice(80, 96, None),), replica_id=0, data=[6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6]),
 Shard(devi

In [6]:
n_local_tokens // x_block_shape[0]

2

In [8]:
n_local_tokens

16

In [22]:
def send_recv_kernel(target_ref, input_ref, output_ref, send_sem, recv_sem):
    my_id = lax.axis_index('x')
    # Can replace with jax.lax.for_i_loop
    for tok_idx in range(target_ref.shape[0]):
        target = target_ref[tok_idx]
        remote_copy_op = pltpu.make_async_remote_copy(
            src_ref=input_ref.at[pl.ds(tok_idx, 1)],
            dst_ref=output_ref.at[pl.ds(tok_idx, 1)],
            send_sem=send_sem,
            recv_sem=recv_sem,
            device_id=(target,),
            device_id_type=pltpu.DeviceIdType.MESH,
        )
        remote_copy_op.start()
        remote_copy_op.wait()


out_shape = jax.ShapeDtypeStruct((n_local_tokens, n_local_col), x.dtype)
grid_spec = pltpu.PrefetchScalarGridSpec(
    num_scalar_prefetch=1,
    in_specs=[
        pl.BlockSpec(memory_space=pltpu.HBM),
    ],
    out_specs=pl.BlockSpec(memory_space=pltpu.HBM),
    scratch_shapes=(
        [pltpu.SemaphoreType.DMA] * 2),
)
right_permute = pl.pallas_call(
    # functools.partial(send_recv_kernel, token_block_size=token_block_size),
    send_recv_kernel,
    out_shape=out_shape,
    grid_spec=grid_spec,
)
# Wrap the kernel within a shard_map to call.
pallas_result = jax.jit(
    jax.shard_map(
        right_permute,
        mesh=mesh,
        in_specs=(P('x'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False,
    )
)(targets, x)

# Compare Pallas result to XLA shard_map result.
perm = tuple((src, (src + 1) % num_devices) for src in range(num_devices))

xla_result = jax.jit(
    jax.shard_map(
        lambda x: lax.ppermute(x, 'x', perm),
        mesh=mesh, in_specs=P('x', None), out_specs=P('x', None))
)(x)

print(pallas_result)

np.testing.assert_array_equal(pallas_result, xla_result)

[[14336 14337 14338 ... 14461 14462 14463]
 [14464 14465 14466 ... 14589 14590 14591]
 [14592 14593 14594 ... 14717 14718 14719]
 ...
 [13952 13953 13954 ... 14077 14078 14079]
 [14080 14081 14082 ... 14205 14206 14207]
 [14208 14209 14210 ... 14333 14334 14335]]


In [12]:
a = jnp.arange(5)
b = jnp.arange(6)
for i in a:
    print(type(i))
    print(i)
    print(i.shape)
print(a[b[3]])

<class 'jaxlib._jax.ArrayImpl'>
0
()
<class 'jaxlib._jax.ArrayImpl'>
1
()
<class 'jaxlib._jax.ArrayImpl'>
2
()
<class 'jaxlib._jax.ArrayImpl'>
3
()
<class 'jaxlib._jax.ArrayImpl'>
4
()
3
